# SAM 3 Forensic Object Detection

Open-vocabulary object detection & segmentation for forensic evidence
triage, built on Meta AI's **SAM 3** via the `ultralytics` SAM3 predictor.

This notebook:
1. Installs dependencies
2. Downloads the gated SAM 3 checkpoint from Hugging Face
3. Loads an image (from Colab upload or a local path)
4. Runs open-vocabulary detection/segmentation using a custom class list
5. Visualizes and saves the annotated results

> This notebook is a **visual triage aid**, not a forensic authentication
> tool. All detections should be reviewed by a qualified examiner. See
> `docs/methodology.md` and `README.md` for scope and limitations.

**Runtime:** Designed for Google Colab with a GPU runtime, but works in any
Jupyter environment with CUDA/MPS/CPU (CPU will be slow).


## 1. Install dependencies

In [ ]:
!pip install -q ultralytics huggingface_hub

## 2. Authenticate with Hugging Face

SAM 3 (`facebook/sam3`) is a gated model. You need a Hugging Face account
with access approved, and a valid access token.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 3. Download the SAM 3 checkpoint

In [ ]:
from huggingface_hub import snapshot_download

snapshot_path = snapshot_download(repo_id="facebook/sam3")

print("Downloaded to:")
print(snapshot_path)

In [ ]:
import os

# Locate the model weight file inside the downloaded snapshot
checkpoint_candidates = []
for root, dirs, files in os.walk(snapshot_path):
    for file in files:
        if file.endswith((".pt", ".pth", ".bin", ".safetensors")):
            checkpoint_candidates.append(os.path.join(root, file))

print("Found checkpoint file(s):")
for c in checkpoint_candidates:
    print(" -", c)

## 4. Copy the checkpoint to a working path

Update `SRC_CHECKPOINT` below if the discovered path differs from the
expected `sam3.pt` file found above.


In [ ]:
import shutil

SRC_CHECKPOINT = checkpoint_candidates[0]  # adjust if needed
DST_CHECKPOINT = "./sam3.pt"

shutil.copy2(SRC_CHECKPOINT, DST_CHECKPOINT)

print("SAM 3 checkpoint ready at:", DST_CHECKPOINT)

### Optional: download the checkpoint locally (Colab only)

Uncomment if you want a local copy of the `.pt` file on your machine (e.g.
to reuse without re-downloading from Hugging Face next time).


In [ ]:
# from google.colab import files
# files.download(DST_CHECKPOINT)

## 5. Load an input image

- In **Colab**: run the upload cell below.
- In a **local Jupyter** environment: skip the upload cell and set
  `image_path` directly to a file in `images/input/`.


In [ ]:
# --- Option A: Colab upload ---
# from google.colab import files
# uploaded = files.upload()
# image_path = "./" + list(uploaded.keys())[0]

# --- Option B: local file ---
image_path = "images/input/example.jpg"  # <-- update to your image

print("Using image:", image_path)

## 6. Load the SAM 3 predictor

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

overrides = {
    "conf": 0.25,
    "task": "segment",
    "mode": "predict",
    "model": DST_CHECKPOINT,
    "save": True,
}

predictor = SAM3SemanticPredictor(overrides=overrides)

print("SAM 3 predictor loaded.")

## 7. Define the class prompt list

SAM 3 is open-vocabulary: you can detect/segment any object you can name in
plain text. Edit this list for your own use case. Short, concrete noun
phrases work best (see `docs/methodology.md`, section 4).


In [ ]:
classes = [
    "laptop",
    "cell phone",
    "external hard drive",
    "USB drive",
    "WiFi router",
    "SD card",
    "memory card",
    "SIM card",
    "camera",
    "DVR",
    "hard drive",
    "SSD",
]

print(f"Testing {len(classes)} classes:")
print(classes)

## 8. Run detection

In [ ]:
predictor.set_image(image_path)

results = predictor(text=classes)

print("Detection completed.")

## 9. Visualize results

In [ ]:
from IPython.display import display
from PIL import Image

plotted_images = []
for result in results:
    plotted = result.plot()
    plotted_rgb = plotted[:, :, ::-1]
    img = Image.fromarray(plotted_rgb)
    plotted_images.append(img)
    display(img)

## 10. Save annotated results

Saves each annotated image to `images/results/` and `outputs/sample_detections/`.


In [ ]:
import os
from datetime import datetime

os.makedirs("images/results", exist_ok=True)
os.makedirs("outputs/sample_detections", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for i, img in enumerate(plotted_images):
    out_path = f"images/results/detection_{timestamp}_{i}.png"
    img.save(out_path)
    print("Saved:", out_path)

    sample_path = f"outputs/sample_detections/detection_{timestamp}_{i}.png"
    img.save(sample_path)

## Notes

- All detections here are **candidates for human review**, not confirmed
  identifications.
- See `README.md` for setup details and `docs/methodology.md` for the full
  methodology, prompt-design guidance, and known limitations.
